# Step 6: Propensity Score and Causal Inference Analysis

**Study:** Synthetic RWE Diabetes Treatment Study  
**Comparison:** Metformin users (Group A) vs Other diabetes medication users (Group B)

**Methods:**
- Pre-treatment confounder selection (demographics + comorbidities)
- Logistic regression propensity score: P(Metformin | X)
- 1:1 nearest-neighbor matching (sklearn) with logit PS caliper
- Standardized mean difference (SMD) balance diagnostics and Love plot
- Matched logistic regression, risk difference, IPTW ATE, IPTW Cox (lifelines)
- Sensitivity analyses (calipers, reduced covariate model)

**Causal disclaimer:** Because this is an observational study, causal interpretation relies on assumptions including no unmeasured confounding and correct model specification.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "synthea_csv"
REPORT_DIR = PROJECT_ROOT / "reports"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from causal_inference import (
    run_causal_inference,
    CONFOUNDER_COLUMNS,
    TREATMENT_A,
    TREATMENT_B,
    SMD_THRESHOLD,
)

## Confounder selection

Baseline covariates measured **before index treatment date**:

| Category | Variables |
| --- | --- |
| Demographics | age, gender (female indicator) |
| Clinical | hypertension, cardiovascular disease, obesity, hyperlipidemia, chronic kidney disease |

SMD &lt; 0.1 indicates acceptable balance after matching or weighting.

In [ ]:
result = run_causal_inference(DATA_DIR, REPORT_DIR)

analysis_data = result.analysis_data
propensity_scores = result.propensity_scores
matched = result.matched_cohort
balance = result.balance
treatment_effects = result.treatment_effects
sensitivity = result.sensitivity

## Propensity scores

In [ ]:
print(f"Analytic cohort N = {len(analysis_data)}")
print(f"{TREATMENT_A}: {(analysis_data['treatment_metformin'] == 1).sum()}")
print(f"{TREATMENT_B}: {(analysis_data['treatment_metformin'] == 0).sum()}")

display(
    propensity_scores.groupby("treatment_group")["propensity_score"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(3)
)

print("\nSample propensity scores:")
display(propensity_scores.head(10))

## Matching results

In [ ]:
n_pairs = matched["pair_id"].nunique() if not matched.empty else 0
print(f"1:1 matched pairs: {n_pairs}")

if not matched.empty:
    display(
        matched[
            ["patient_id", "treatment_group", "propensity_score", "pair_id", "match_distance_logit_ps", "outcome"]
        ].sort_values(["pair_id", "role"]).head(16)
    )

## Balance assessment (SMD before vs after matching)

In [ ]:
pivot = balance.pivot(index="covariate", columns="analysis_stage", values="smd")
display(pivot.round(3))

after = balance[balance["analysis_stage"] == "After matching"]
if not after.empty:
    pct = after["balanced"].mean() * 100
    print(f"\nCovariates balanced after matching (|SMD| < {SMD_THRESHOLD}): {pct:.0f}%")

## Treatment effect estimation

In [ ]:
display(treatment_effects)

print("\nSensitivity analyses:")
display(sensitivity)

## Visualizations

In [ ]:
from IPython.display import Image, display

for fig_path in result.figure_paths:
    print(fig_path.name)
    display(Image(filename=str(fig_path)))

## Interpretation

Adjusted analyses reduce but do not eliminate confounding by indication. With this small Synthea cohort, wide confidence intervals and limited matched sample size constrain precision.

**Outputs:**
- `reports/propensity_scores.csv`
- `reports/matched_cohort.csv`
- `reports/balance_diagnostics.csv`
- `reports/treatment_effect_results.csv`
- `reports/causal_sensitivity_analysis.csv`
- `reports/causal_analysis_report.html`

Because this is an observational study, causal interpretation relies on assumptions including no unmeasured confounding and correct model specification.